In [ ]:
# %% [markdown]
# # 01 — Exploratory Data Analysis (EDA)
# ## Online Retail II Dataset
# **RetailPulse — Zidio Development | March 2026**


In [ ]:
# %% [markdown]
# ## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# %% [markdown]
# ## Load Dataset

In [ ]:
df = pd.read_excel('../data/raw/online_retail_ii/online_retail_II.xlsx')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.info(show_counts=True))

In [ ]:
# %% [markdown]
# ## Data Cleaning

In [ ]:
print(f"Before cleaning: {df.shape}")

# Remove cancelled
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int)
df['TotalAmount'] = df['Quantity'] * df['Price']
df = df.drop_duplicates()

print(f"After cleaning: {df.shape}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique invoices: {df['Invoice'].nunique():,}")
print(f"Unique products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique()}")

In [ ]:
# %%[markdown]
# ## Missing Values Matrix

In [ ]:
msno.matrix(df)
plt.title("Missing Values Matrix — Online Retail II")
plt.tight_layout()
plt.savefig('../reports/missing_values.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%[markdown]
# ## Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['Quantity'].hist(ax=axes[0], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Quantity Distribution')
axes[0].set_xlabel('Quantity')
df['Price'].hist(ax=axes[1], bins=50, color='coral', edgecolor='black')
axes[1].set_title('Price Distribution')
axes[1].set_xlabel('Price (£)')
plt.tight_layout()
plt.savefig('../reports/distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%[markdown]
# ## Correlation Heatmap

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Hour'] = df['InvoiceDate'].dt.hour

numeric_df = df[['Quantity', 'Price', 'TotalAmount', 'Month', 'DayOfWeek']].dropna()
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', bbox_inches='tight')
plt.show()

print(df[['Quantity', 'Price', 'TotalAmount', 'Month', 'DayOfWeek']].describe())

In [ ]:
# %%[markdown]
# ## Top Products by Revenue

In [ ]:
top_products = df.groupby('Description')['TotalAmount'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
top_products.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Revenue (£)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/top_products.png', bbox_inches='tight')
plt.show()

In [ ]:
# %%[markdown]
# ## Revenue by Country

In [ ]:
top_countries = df.groupby('Country')['TotalAmount'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
top_countries.plot(kind='bar', color='orange', edgecolor='black')
plt.title('Revenue by Country (Top 10)')
plt.ylabel('Total Revenue (£)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/revenue_by_country.png', bbox_inches='tight')
plt.show()

print(f"Total revenue: £{df['TotalAmount'].sum():,.2f}")
print(f"Avg order value: £{df['TotalAmount'].mean():.2f}")
print(f"Total transactions: {len(df):,}")